# Tìm hiểu kiến trúc mô hình OWL-ViT và Grounding DINO cho bài toán Object Detection bằng Text-Prompted trên dữ liệu ảnh thời sự Việt Nam
### Faster R-CNN (closed-set) · OWL-ViT v2 · Grounding DINO (open-vocabulary)

---

**Môn học:** Học sâu (Deep Learning)  
**Trường:** Đại học Công Nghệ Kỹ thuật TP.HCM (HCMUTE)  
**Nhóm thực hiện:** Nhóm 2
- Nguyễn Thanh Quang — 23110142  
- Trịnh Nhật Anh — 23110074  
- Đặng Quốc Thành Tài — 23110149  

**Giảng viên hướng dẫn:** Lê Minh Tân  
**Thời gian:** 
Học kỳ II
Năm học: 2025 - 2026

---

## 1. Giới thiệu

Notebook này đánh giá và so sánh ba mô hình phát hiện đối tượng (object detection)
đại diện cho ba thế hệ và triết lý thiết kế khác nhau, trên domain **ảnh thời sự
Việt Nam** (dữ liệu AIC 2025). Mục tiêu là khảo sát **domain gap**: các mô hình
open-vocabulary được huấn luyện trên dữ liệu phương Tây có còn hiệu quả khi áp
dụng cho dữ liệu tin tức Việt Nam hay không.

## 2. Ba mô hình so sánh

| Mô hình | Loại | Backbone | Tham số | Vai trò |
|---|---|---|---|---|
| Faster R-CNN | Closed-set (2015) | ResNet (OpenImagesV4) | — | Baseline |
| OWL-ViT v2 | Open-vocab, fusion nông | ViT-B/16 (CLIP) | ~154M | So sánh |
| Grounding DINO | Open-vocab, deep fusion | Swin-T + BERT | ~170M | So sánh |

- **Faster R-CNN:** kết quả tiền tính sẵn từ TF Hub (OpenImagesV4).
- **OWL-ViT v2:** checkpoint `google/owlv2-base-patch16-ensemble`.
- **Grounding DINO:** checkpoint `IDEA-Research/grounding-dino-tiny`.

## 3. Dữ liệu

- **Nguồn:** AIC 2025 (vòng Sơ tuyển) — keyframe trích từ video tin tức Việt Nam.
- **Tập kiểm thử:** 151 ảnh, gán nhãn thủ công với 13 lớp đối tượng.
- **Ground truth:** xây dựng bán tự động (tiền-gán-nhãn từ Faster R-CNN + hiệu chỉnh tay), export định dạng COCO.

## 4. Phương pháp đánh giá

- **Metric:** mAP theo chuẩn COCO (mAP@[.5:.95], mAP@50, mAP@75) + Average Recall.
- **Công cụ:** pycocotools.
- **Inference:** mỗi mô hình chạy với 13 câu truy vấn (text query) trên từng ảnh.

## 5. Môi trường

- **GPU:** NVIDIA RTX Pro 6000 (Kaggle).
- **Ràng buộc:** internet tắt khi dùng GPU → model và dữ liệu nạp offline từ Kaggle Dataset.
- **Thư viện chính:** PyTorch, HuggingFace Transformers, pycocotools.

## 6. Cấu trúc notebook

1. Cấu hình đường dẫn & nạp ground truth  
2. Baseline: Faster R-CNN  
3. Nạp mô hình OWL-ViT v2 & Grounding DINO (offline)  
4. Chạy inference  
5. Tính mAP 3-way (tổng + theo lớp)  
6. Trực quan hoá các trường hợp lỗi

# 1. Cấu Hình Đường Dẫn & Nạp Ground Truth

## 1.1. Cấu hình đường dẫn

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"

# Ảnh Ground truth
GT_DIR  = "/kaggle/input/datasets/trinhnhatanh/vn-news/test"
GT      = f"{GT_DIR}/_annotations.coco.json"
IMG_DIR = GT_DIR

# Faster R-CNN
OBJ_JSONL = "/kaggle/input/datasets/trinhnhatanh/aic-test-2025"

# model offline
MODELS  = "/kaggle/input/datasets/trinhnhatanh/dl-model/models"

## 1.2. Nạp Ground Truth

In [ ]:
import json
gt = json.load(open(GT))
id2name = {c["id"]:c["name"] for c in gt["categories"]}
name2id = {c["name"]:c["id"] for c in gt["categories"]}

def name_to_key(name):
    base = name.replace(".jpg",""); vid,kf = base.rsplit("_",1); return f"{vid}/{kf}"
imgid2key  = {im["id"]: name_to_key(im["extra"]["name"]) for im in gt["images"]}
key2imgid  = {v:k for k,v in imgid2key.items()}
imgid2wh   = {im["id"]:(im["width"],im["height"]) for im in gt["images"]}
imgid2file = {im["id"]: f"{IMG_DIR}/{im['file_name']}" for im in gt["images"]}

GT2QUERY = {"building":"a building","drink":"a drink","flower":"a flower","food":"food",
    "human face":"a human face","man":"a man","person":"a person","poster":"a poster",
    "table":"a table","tree":"a tree","vehicle":"a vehicle","window":"a window","woman":"a woman"}
QUERIES = [(g,q,name2id[g]) for g,q in GT2QUERY.items() if g in name2id]
print(f"{len(gt['images'])} ảnh, {len(gt['annotations'])} GT box, {len(QUERIES)} query")

# 2. Baseline: Faster R-CNN

In [ ]:
ENT2GT = {"Building":"building","Drink":"drink","Flower":"flower","Food":"food",
    "Human face":"human face","Man":"man","Person":"person","Poster":"poster",
    "Table":"table","Tree":"tree","Vehicle":"vehicle","Land vehicle":"vehicle",
    "Window":"window","Woman":"woman"}

frcnn = {}
with open(f"{OBJ_JSONL}/objects_test.jsonl") as f:
    for line in f:
        r = json.loads(line)
        if r["key"] in key2imgid: frcnn[r["key"]] = r

preds_frcnn = []
for key, r in frcnn.items():
    img_id = key2imgid[key]; W,H = imgid2wh[img_id]
    scores = [float(s) for s in r["detection_scores"]]
    for box, ent, sc in zip(r["detection_boxes"], r["detection_class_entities"], scores):
        if sc < 0.05 or ent not in ENT2GT or ENT2GT[ent] not in name2id: continue
        ymin,xmin,ymax,xmax = [float(v) for v in box]
        x,y,w,h = xmin*W, ymin*H, (xmax-xmin)*W, (ymax-ymin)*H
        if w<=1 or h<=1: continue
        preds_frcnn.append({"image_id":img_id,"category_id":name2id[ENT2GT[ent]],
                            "bbox":[round(x,2),round(y,2),round(w,2),round(h,2)],"score":sc})
print(f"Faster R-CNN: {len(preds_frcnn)} predictions trên {len(frcnn)} ảnh")

# 3. Nạp mô hình OWL-ViT v2 & Grounding DINO (offline)

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"]="1"; os.environ["TRANSFORMERS_OFFLINE"]="1"
import torch, time
from transformers import (Owlv2Processor, Owlv2ForObjectDetection,
                          AutoProcessor, AutoModelForZeroShotObjectDetection)

device = "cuda" if torch.cuda.is_available() else "cpu"
MODELS = "/kaggle/input/datasets/trinhnhatanh/dl-model/models"        # sửa nếu path khác

owl_p = Owlv2Processor.from_pretrained(f"{MODELS}/owlv2")
owl_m = Owlv2ForObjectDetection.from_pretrained(f"{MODELS}/owlv2").to(device).eval()
gd_p  = AutoProcessor.from_pretrained(f"{MODELS}/gdino")
gd_m  = AutoModelForZeroShotObjectDetection.from_pretrained(f"{MODELS}/gdino").to(device).eval()
print("Loaded offline OK:", device)

# wrapper chịu được nhiều version transformers (threshold vs box_threshold)
def _owl_post(out, ts, q, thr):
    fn = owl_p.post_process_grounded_object_detection
    try:    return fn(outputs=out, threshold=thr, target_sizes=ts, text_labels=[[q]])[0]
    except TypeError:
        try:    return fn(outputs=out, threshold=thr, target_sizes=ts)[0]
        except TypeError: return owl_p.post_process_object_detection(out,thr,ts)[0]

def _gd_post(out, ids, ts, bt, tt):
    fn = gd_p.post_process_grounded_object_detection
    try:    return fn(out, ids, threshold=bt, text_threshold=tt, target_sizes=ts)[0]
    except TypeError: return fn(out, ids, box_threshold=bt, text_threshold=tt, target_sizes=ts)[0]

def owl_detect(image, query, thr=0.1):
    inp = owl_p(text=[[query]], images=image, return_tensors="pt").to(device)
    with torch.no_grad(): out = owl_m(**inp)
    ts = torch.tensor([(image.height,image.width)]).to(device)
    r = _owl_post(out, ts, query, thr)
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

def gd_detect(image, query, bt=0.25, tt=0.2):
    inp = gd_p(images=image, text=query.lower()+".", return_tensors="pt").to(device)
    with torch.no_grad(): out = gd_m(**inp)
    r = _gd_post(out, inp.input_ids, [(image.height,image.width)], bt, tt)
    return r["boxes"].cpu().numpy(), r["scores"].cpu().numpy()

# 4. Chạy inference OWL-ViT + Grounding DINO

In [ ]:
from PIL import Image
def xyxy_to_xywh(b): return [round(float(b[0]),2),round(float(b[1]),2),
                             round(float(b[2]-b[0]),2),round(float(b[3]-b[1]),2)]

def run_model(detect_fn, name):
    preds, t0 = [], time.time()
    for i, im in enumerate(gt["images"]):
        img = Image.open(imgid2file[im["id"]]).convert("RGB")
        for gtname, query, cid in QUERIES:
            boxes, scores = detect_fn(img, query)
            for b, s in zip(boxes, scores):
                xywh = xyxy_to_xywh(b)
                if xywh[2]>1 and xywh[3]>1:
                    preds.append({"image_id":im["id"],"category_id":cid,
                                  "bbox":xywh,"score":float(s)})
        if (i+1) % 30 == 0: print(f"  {name}: {i+1}/{len(gt['images'])} ({time.time()-t0:.0f}s)")
    print(f"{name}: {len(preds)} preds, {time.time()-t0:.0f}s ({(time.time()-t0)/len(gt['images']):.2f}s/ảnh)")
    return preds

preds_owl = run_model(owl_detect, "OWL-ViT")
preds_gd  = run_model(gd_detect,  "G-DINO")
json.dump({"frcnn":preds_frcnn, "owl":preds_owl, "gd":preds_gd},
          open("/kaggle/working/all_predictions.json","w"))

# 5. Tính mAP 3-way (tổng + theo lớp)

## Theo Tổng

In [ ]:
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
import numpy as np
coco_gt = COCO(GT)

def evaluate(preds, name):
    if not preds: print(f"{name}: rỗng"); return
    dt = coco_gt.loadRes(preds)
    E = COCOeval(coco_gt, dt, "bbox"); E.evaluate(); E.accumulate()
    print(f"\n===== {name} ====="); E.summarize()
    # mAP theo từng lớp (mAP@50)
    print(f"  mAP@50 theo lớp:")
    for cid in coco_gt.getCatIds():
        if cid == 0: continue
        E2 = COCOeval(coco_gt, dt, "bbox"); E2.params.catIds=[cid]
        E2.evaluate(); E2.accumulate(); E2.summarize_=lambda:None
        prec = E2.eval["precision"][0,:,0,0,2]  # IoU=0.5
        ap = prec[prec>-1].mean() if (prec>-1).any() else 0
        print(f"    {id2name[cid]:14s} {ap:.3f}")
    return E.stats

for preds, name in [(preds_frcnn,"Faster R-CNN"),(preds_owl,"OWL-ViT"),(preds_gd,"Grounding DINO")]:
    evaluate(preds, name)

## Theo Lớp

In [ ]:
import numpy as np
from pycocotools.cocoeval import COCOeval

def per_class_ap50(preds, name):
    dt = coco_gt.loadRes(preds)
    E = COCOeval(coco_gt, dt, "bbox"); E.evaluate(); E.accumulate()
    print(f"\n{name} — AP@50 theo lớp:")
    for ci, cid in enumerate(coco_gt.getCatIds()):
        if cid == 0: continue
        p = E.eval["precision"][0, :, ci, 0, 2]
        ap = p[p > -1].mean() if (p > -1).any() else float("nan")
        print(f"  {id2name[cid]:14s} {ap:.3f}")
    return E.stats

for preds, name in [(preds_frcnn,"Faster R-CNN"),(preds_owl,"OWL-ViT"),(preds_gd,"Grounding DINO")]:
    per_class_ap50(preds, name)

## Tính Lại mAP Khi Bỏ Lớp "poster"

In [ ]:
from pycocotools.cocoeval import COCOeval

poster_id = name2id["poster"]
real_ids   = [c for c in coco_gt.getCatIds() if c != 0]
no_poster  = [c for c in real_ids if c != poster_id]

def map_summary(preds, catids):
    dt = coco_gt.loadRes(preds)
    E = COCOeval(coco_gt, dt, "bbox")
    E.params.catIds = catids
    E.evaluate(); E.accumulate()
    E.summarize_ = None
    import io, contextlib
    with contextlib.redirect_stdout(io.StringIO()):
        E.summarize()
    return E.stats[0], E.stats[1], E.stats[2]

print(f"{'Model':16s} {'mAP@50:95':>10s} {'mAP@50':>8s} {'mAP@75':>8s}")
print("-"*46)
for preds, name in [(preds_frcnn,"Faster R-CNN"),(preds_owl,"OWL-ViT"),(preds_gd,"Grounding DINO")]:
    a, b, c = map_summary(preds, no_poster)
    print(f"{name:16s} {a:>10.3f} {b:>8.3f} {c:>8.3f}")
print("\n(Đã loại lớp 'poster' do annotation không nhất quán)")

# 6. Visualize 1 Lớp Cụ Thể Của 1 Model Vs Ground Truth

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from collections import defaultdict

def viz_class(preds, target_gtname, n=3, score_min=0.2):
    cid = name2id[target_gtname]
    # gom prediction theo ảnh cho lớp này
    pred_by_img = defaultdict(list)
    for p in preds:
        if p["category_id"]==cid and p["score"]>=score_min:
            pred_by_img[p["image_id"]].append(p)
    # gom GT theo ảnh
    gt_by_img = defaultdict(list)
    for a in gt["annotations"]:
        if a["category_id"]==cid: gt_by_img[a["image_id"]].append(a)
    # chọn ảnh có cả GT lẫn pred để so sánh
    ids = [i for i in gt_by_img if i in pred_by_img][:n]
    fig, axes = plt.subplots(1, len(ids), figsize=(6*len(ids),5))
    if len(ids)==1: axes=[axes]
    for ax, img_id in zip(axes, ids):
        ax.imshow(Image.open(imgid2file[img_id]).convert("RGB"))
        for a in gt_by_img[img_id]:        # GT = xanh lá
            x,y,w,h=a["bbox"]; ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,color="lime",lw=2))
        for p in pred_by_img[img_id]:      # pred = đỏ
            x,y,w,h=p["bbox"]; ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,color="red",lw=2))
            ax.text(x,y-3,f'{p["score"]:.2f}',color="red")
        ax.set_title(f'{target_gtname} (xanh=GT, đỏ=dự đoán)'); ax.axis("off")
    plt.tight_layout(); plt.savefig(f"/kaggle/working/viz_{target_gtname.replace(' ','_')}.png",dpi=100); plt.show()

viz_class(preds_owl, "human face")
viz_class(preds_frcnn, "poster")    
viz_class(preds_gd, "poster")